In [ ]:
"""
=============================================================================
03_zero_shot_pipeline.py
ZERO-SHOT PIPELINE — KLASIFIKASI BAHAYA K3 PERTAMBANGAN
=============================================================================
Thesis : Evaluasi Komparatif Metode Klasifikasi Pendekatan LLM
         pada Data Keselamatan Industri
Penulis: Riza Rahmah Angelia | NRP: M0501241008 | IPB University
=============================================================================

PREREQUISITE:
  Jalankan 00_pipeline_eda_preprocessing.py terlebih dahulu.
  File yang dibutuhkan:
    • split_test_set.csv         → 71 baris, Test Set FIXED
    • rag_knowledge_docs.json    → 28 dokumen definisi + keywords TBC

PERBEDAAN DENGAN PENDEKATAN RAG (Notebook 02):
  • TIDAK ada retrieval (ChromaDB tidak digunakan)
  • TIDAK ada few-shot contoh dari Knowledge Base
  • Seluruh definisi kategori + keywords di-inject langsung ke prompt
  • Parameter ablasi: hanya Temperature (T) karena tidak ada TOP_K
  • Lebih efisien per-prediksi, lebih panjang per-prompt

ALUR PIPELINE — PENDEKATAN ZERO-SHOT:
  ┌────────────────────────────────────────────────────────────────┐
  │  TAHAP 0 → Import & Konfigurasi                               │
  │  TAHAP 1 → Load Data & Bangun Prompt Template                 │
  │             Inject semua definisi + keywords dari JSON        │
  │                                                               │
  │  TAHAP 2 — Ablasi Temperature (3 nilai)                       │
  │    T = 0.1, 0.3, 0.5  (1x run per T)                        │
  │    Metrik seleksi: F1-Macro                                   │
  │    → Pilih T terbaik                                          │
  │                                                               │
  │  TAHAP 3 — Stability Test 3x run (T terbaik)                 │
  │    → Hitung κ Stability                                       │
  │                                                               │
  │  TAHAP 4 — Uji Kestabilan Final (5x run)                     │
  │    → κ Reliability + κ Stability resmi untuk thesis          │
  │                                                               │
  │  TAHAP 5 — Visualisasi & Ekspor                              │
  └────────────────────────────────────────────────────────────────┘

ARSITEKTUR ZERO-SHOT:
  LLM        : google/flan-t5-large (lokal, gratis)
  Prompt     : Zero-shot — definisi + keywords dari rag_knowledge_docs.json
  Retrieval  : TIDAK ADA (tidak menggunakan ChromaDB)

INSTALASI:
=============================================================================
"""
#!pip install transformers torch scikit-learn pandas numpy matplotlib seaborn tqdm openpyxl

In [ ]:
# =============================================================================
# TAHAP 0: IMPORT & KONFIGURASI
# =============================================================================
import os
import json
import warnings
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from collections import Counter
from itertools import combinations
from datetime import datetime
from tqdm import tqdm

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, cohen_kappa_score,
    confusion_matrix, classification_report,
)

warnings.filterwarnings("ignore")
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor"  : "#f8f9fa",
    "axes.grid"       : True,
    "grid.alpha"      : 0.4,
    "font.family"     : "sans-serif",
    "font.size"       : 11,
})

# ── Path input ──────────────────────────────────────────────────────────────
TEST_PATH      = "split_test_set.csv"
KNOWLEDGE_FILE = "rag_knowledge/rag_knowledge_docs.json"

# ── Path output ─────────────────────────────────────────────────────────────
OUTPUT_DIR = "output_zeroshot"
CACHE_FILE = f"{OUTPUT_DIR}/run_cache.json"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Model ────────────────────────────────────────────────────────────────────
LLM_MODEL         = "google/flan-t5-large"
DEVICE            = "cuda" if torch.cuda.is_available() else "cpu"
MAX_SOURCE_LENGTH = 1024
MAX_TARGET_LENGTH = 64

# ── Kolom ────────────────────────────────────────────────────────────────────
COL_LABEL = "label"
COL_TEXT  = "teks_final"

# ── Ruang pencarian hyperparameter (hanya Temperature) ───────────────────────
KANDIDAT_T       = [0.1, 0.3, 0.5]
N_STABILITY_RUNS = 3
N_FINAL_RUNS     = 5

print("  ZERO-SHOT PIPELINE")
print(f"  LLM    : {LLM_MODEL}")
print(f"  Device : {DEVICE}")
print(f"  Ruang T: {KANDIDAT_T}")

In [ ]:
# =============================================================================
# TAHAP 1: LOAD DATA & BANGUN PROMPT TEMPLATE
# =============================================================================
print("TAHAP 1: LOAD DATA & BANGUN PROMPT TEMPLATE")

# ── 1a. Load test set ────────────────────────────────────────────────────────
df_test = pd.read_csv(TEST_PATH)
df_test[COL_TEXT]  = df_test[COL_TEXT].fillna("").astype(str)
df_test[COL_LABEL] = df_test[COL_LABEL].astype(str).str.strip()

GROUND_TRUTH = df_test[COL_LABEL].tolist()
N_TEST       = len(df_test)
all_labels   = sorted(df_test[COL_LABEL].unique())

print(f"\n✔ Test set : {N_TEST} dokumen")
print(f"✔ Kelas   : {len(all_labels)}")

# ── 1b. Load knowledge docs & bangun konten prompt ───────────────────────────
with open(KNOWLEDGE_FILE, encoding="utf-8") as f:
    tbc_docs = json.load(f)

# Pisahkan definisi dan keywords_severity
defs_by_label = {}   # label → teks definisi
kw_by_label   = {}   # label → teks keywords_severity
for doc in tbc_docs:
    lbl = doc["label"]
    if doc["type"] == "definition":
        defs_by_label[lbl] = doc["text"]
    elif doc["type"] == "keywords_severity":
        kw_by_label[lbl] = doc["text"]

# ── 1c. Bangun {manual_rules} — aturan standar evaluasi ──────────────────────
# Berisi definisi setiap kategori (apa saja temuan yang termasuk)
manual_rules_parts = []
for lbl in sorted(defs_by_label.keys()):
    manual_rules_parts.append(defs_by_label[lbl])
MANUAL_RULES = "\n\n".join(manual_rules_parts)

# ── 1d. Bangun {rag_knowledge_docs_definitions} — definisi + keywords ─────────
# Berisi keywords dan indikator severity per kategori
knowledge_parts = []
for lbl in sorted(kw_by_label.keys()):
    knowledge_parts.append(kw_by_label[lbl])
KNOWLEDGE_DEFS = "\n\n".join(knowledge_parts)

# ── 1e. Bangun {list_of_valid_categories} ─────────────────────────────────────
VALID_CATEGORIES = "\n".join(f"- {lbl}" for lbl in all_labels)

print(f"\n✔ Definisi kategori dimuat  : {len(defs_by_label)} kategori")
print(f"✔ Keywords severity dimuat  : {len(kw_by_label)} kategori")
print(f"\nDaftar kategori valid:")
for lbl in all_labels:
    print(f"  • {lbl}")

# ── 1f. Prompt template (zero-shot) ──────────────────────────────────────────
PROMPT_TEMPLATE = """Anda adalah Ahli Evaluator Keselamatan K3 Pertambangan. Tugas Anda adalah mengklasifikasikan deskripsi laporan insiden atau bahaya operasional ke dalam SATU kategori baku yang paling tepat.

Anda harus mengandalkan kemampuan penalaran logis Anda secara mandiri berdasarkan aturan dan definisi standar keselamatan di bawah ini.

<aturan_standar_evaluasi>
{manual_rules}
</aturan_standar_evaluasi>

<definisi_dan_kata_kunci_kategori>
{rag_knowledge_docs_definitions}
</definisi_dan_kata_kunci_kategori>

<daftar_kategori_valid>
{list_of_valid_categories}
</daftar_kategori_valid>

INSTRUKSI OUTPUT:
1. Baca teks laporan bahaya di bawah ini secara saksama untuk mencari akar masalah utama (root cause).
2. Petakan akar masalah tersebut ke dalam <aturan_standar_evaluasi> dan <definisi_dan_kata_kunci_kategori>.
3. Tentukan satu kategori yang paling mutlak dan akurat dari <daftar_kategori_valid>.
4. Jawab HANYA dengan nama kategori yang valid tanpa tambahan teks, tanda kutip, tanda baca, sapaan, atau penjelasan apa pun.

<laporan_bahaya>
{deskripsi_input}
</laporan_bahaya>"""

# Preview prompt untuk satu contoh
sample_text = df_test[COL_TEXT].iloc[0]
sample_prompt = PROMPT_TEMPLATE.format(
    manual_rules                  = MANUAL_RULES,
    rag_knowledge_docs_definitions= KNOWLEDGE_DEFS,
    list_of_valid_categories      = VALID_CATEGORIES,
    deskripsi_input               = sample_text,
)
print(f"\n── Contoh prompt (truncated) ──")
print(sample_prompt[:500] + "\n...")
print(f"\n  Panjang prompt contoh : {len(sample_prompt)} karakter")
print(f"  Token estimasi        : ~{len(sample_prompt)//4} token (1 token ≈ 4 karakter)")
print(f"  MAX_SOURCE_LENGTH     : {MAX_SOURCE_LENGTH} token (akan di-truncate jika melebihi)")

In [ ]:
# =============================================================================
# FUNGSI INTI
# =============================================================================

def build_zero_shot_prompt(query_text: str) -> str:
    """Bangun prompt zero-shot dengan seluruh definisi + keywords di-inject."""
    return PROMPT_TEMPLATE.format(
        manual_rules                   = MANUAL_RULES,
        rag_knowledge_docs_definitions = KNOWLEDGE_DEFS,
        list_of_valid_categories       = VALID_CATEGORIES,
        deskripsi_input                = query_text,
    )


def parse_label(raw: str, labels: list, fallback: str) -> str:
    """Cocokkan output LLM ke label yang valid — pilih substring terpanjang."""
    raw_lower = raw.lower().strip()
    best, best_len = None, 0
    for lbl in labels:
        if lbl.lower() in raw_lower and len(lbl) > best_len:
            best_len = len(lbl)
            best     = lbl
    return best if best else fallback


def compute_metrics(y_true: list, y_pred: list) -> dict:
    """Hitung semua metrik klasifikasi untuk satu run."""
    return {
        "accuracy"    : round(accuracy_score(y_true, y_pred), 4),
        "precision_w" : round(precision_score(y_true, y_pred, average="weighted", zero_division=0), 4),
        "precision_m" : round(precision_score(y_true, y_pred, average="macro",    zero_division=0), 4),
        "recall_w"    : round(recall_score(   y_true, y_pred, average="weighted", zero_division=0), 4),
        "recall_m"    : round(recall_score(   y_true, y_pred, average="macro",    zero_division=0), 4),
        "f1_w"        : round(f1_score(       y_true, y_pred, average="weighted", zero_division=0), 4),
        "f1_m"        : round(f1_score(       y_true, y_pred, average="macro",    zero_division=0), 4),
    }


def majority_vote(all_preds: list) -> list:
    """Majority vote dari beberapa run."""
    return [Counter([r[i] for r in all_preds]).most_common(1)[0][0]
            for i in range(len(all_preds[0]))]


def interpret_kappa(k: float) -> str:
    if k >= 0.80: return "Sangat Kuat"
    if k >= 0.60: return "Kuat"
    if k >= 0.40: return "Sedang"
    if k >= 0.20: return "Lemah"
    return "Sangat Lemah"


# ── Cache global ─────────────────────────────────────────────────────────────
cache: dict = {}
if os.path.exists(CACHE_FILE):
    with open(CACHE_FILE) as f:
        cache = json.load(f)
    print(f"✔ Cache dimuat: {len(cache)} run tersimpan")
else:
    print("  Cache kosong — memulai dari awal")


def save_cache():
    with open(CACHE_FILE, "w") as f:
        json.dump(cache, f, ensure_ascii=False)


def run_single(temperature: float, tokenizer, model,
               run_id: str = "") -> list:
    """
    Inferensi satu run (N_TEST prediksi) dengan temperature tertentu.
    Zero-shot: tidak ada retrieval, seluruh definisi ada di prompt.
    """
    cache_key = f"ZS_T{temperature}_{run_id}"
    if cache_key in cache:
        print(f"  ↩ Cache hit: {cache_key}")
        return cache[cache_key]

    preds = []
    fallback = all_labels[0]  # fallback ke kelas pertama jika parse gagal

    for query_text in tqdm(
        df_test[COL_TEXT].tolist(),
        desc=f"  T={temperature} [{run_id}]",
        ncols=72,
    ):
        prompt = build_zero_shot_prompt(query_text)
        inputs = tokenizer(
            prompt, return_tensors="pt",
            truncation=True, max_length=MAX_SOURCE_LENGTH,
        ).to(DEVICE)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=MAX_TARGET_LENGTH,
                do_sample=True,
                temperature=temperature,
            )

        raw  = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()
        pred = parse_label(raw, all_labels, fallback)
        preds.append(pred)

    cache[cache_key] = preds
    save_cache()
    return preds


# ── Load LLM ─────────────────────────────────────────────────────────────────
print("─" * 65)
print(f"LOAD LLM: {LLM_MODEL}")
print("─" * 65)
tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)
llm_model = AutoModelForSeq2SeqLM.from_pretrained(LLM_MODEL).to(DEVICE)
llm_model.eval()
print(f"✔ Model siap di {DEVICE}")

In [ ]:
# =============================================================================
# TAHAP 2: ABLASI TEMPERATURE (1x run per T)
# =============================================================================
print("═" * 65)
print("TAHAP 2 — ABLASI TEMPERATURE (ZERO-SHOT)")
print(f"  T    : {KANDIDAT_T}")
print(f"  Total prediksi: {len(KANDIDAT_T)} × {N_TEST} = {len(KANDIDAT_T)*N_TEST}")
print("═" * 65)

ablasi_rows = []

for t in KANDIDAT_T:
    print(f"\n── Temperature = {t} ──")
    preds   = run_single(t, tokenizer, llm_model, run_id="ablasi")
    metrics = compute_metrics(GROUND_TRUTH, preds)
    ablasi_rows.append({"temperature": t, "preds": preds, **metrics})
    print(f"  ✔ Acc={metrics['accuracy']:.4f}  F1-M={metrics['f1_m']:.4f}  "
          f"F1-W={metrics['f1_w']:.4f}")

ablasi_df   = pd.DataFrame([{k: v for k, v in r.items() if k != "preds"}
                             for r in ablasi_rows])
BEST_T      = float(ablasi_df.loc[ablasi_df["f1_m"].idxmax(), "temperature"])
BEST_PREDS  = next(r["preds"] for r in ablasi_rows if r["temperature"] == BEST_T)

print(f"\n{'='*65}")
print(f"HASIL ABLASI:")
print(ablasi_df[["temperature","accuracy","f1_m","f1_w",
                 "precision_m","recall_m"]].to_string(index=False))
print(f"\n  ✔ Temperature terbaik: T={BEST_T}  "
      f"(F1-Macro={ablasi_df['f1_m'].max():.4f})")

# Simpan hasil ablasi
ablasi_df.to_excel(f"{OUTPUT_DIR}/ZS_ablasi_temperature.xlsx", index=False)
print(f"  ✔ Disimpan: ZS_ablasi_temperature.xlsx")

In [ ]:
# =============================================================================
# TAHAP 3: STABILITY TEST (3x run — T terbaik)
# =============================================================================
print("═" * 65)
print(f"TAHAP 3 — STABILITY TEST: {N_STABILITY_RUNS}x run")
print(f"  T = {BEST_T}")
print("═" * 65)

stab_preds = []
for run_n in range(1, N_STABILITY_RUNS + 1):
    print(f"\n  ── Run {run_n}/{N_STABILITY_RUNS} ──")
    preds = run_single(BEST_T, tokenizer, llm_model, run_id=f"stab_r{run_n}")
    stab_preds.append(preds)
    m = compute_metrics(GROUND_TRUTH, preds)
    print(f"  ✔ Acc={m['accuracy']:.4f}  F1-M={m['f1_m']:.4f}")

print(f"\nCohen's Kappa Stability (pairwise {N_STABILITY_RUNS} run):")
kappa_vals = []
for r1, r2 in combinations(range(N_STABILITY_RUNS), 2):
    k   = cohen_kappa_score(stab_preds[r1], stab_preds[r2])
    lbl = f"R{r1+1}–R{r2+1}"
    kappa_vals.append(k)
    print(f"  {lbl}: κ = {k:.4f}  ({interpret_kappa(k)})")

stab_mean = np.mean(kappa_vals)
stab_std  = np.std(kappa_vals)
print(f"\n  κ Stability rata-rata: {stab_mean:.4f} ± {stab_std:.4f}  "
      f"({interpret_kappa(stab_mean)})")

In [ ]:
# =============================================================================
# TAHAP 4: UJI KESTABILAN FINAL (5x run)
# =============================================================================
print("═" * 65)
print(f"TAHAP 4 — UJI KESTABILAN FINAL: {N_FINAL_RUNS}x run")
print(f"  T = {BEST_T}")
print(f"  Total prediksi: {N_FINAL_RUNS} × {N_TEST} = {N_FINAL_RUNS * N_TEST}")
print("═" * 65)

final_all_preds = []
final_logs      = []

for run_n in range(1, N_FINAL_RUNS + 1):
    print(f"\n  ── Run {run_n}/{N_FINAL_RUNS} ──")
    preds = run_single(BEST_T, tokenizer, llm_model, run_id=f"final_r{run_n}")
    final_all_preds.append(preds)
    m = compute_metrics(GROUND_TRUTH, preds)
    for i, (pred, true) in enumerate(zip(preds, GROUND_TRUTH)):
        final_logs.append({
            "run": run_n, "temperature": BEST_T,
            "test_idx": i, "true_label": true, "pred_label": pred,
            "correct": pred == true,
        })
    print(f"  ✔ Acc={m['accuracy']:.4f}  F1-M={m['f1_m']:.4f}  F1-W={m['f1_w']:.4f}")

# Majority vote
final_preds = majority_vote(final_all_preds)
mv_final    = compute_metrics(GROUND_TRUTH, final_preds)

# κ Reliability
kappa_rel   = cohen_kappa_score(GROUND_TRUTH, final_preds)

# κ Stability pairwise
kappa_pairs_val = []
kappa_pairs_lbl = []
print(f"\nCohen's Kappa Stability (pairwise {N_FINAL_RUNS} run):")
for r1, r2 in combinations(range(N_FINAL_RUNS), 2):
    k   = cohen_kappa_score(final_all_preds[r1], final_all_preds[r2])
    lbl = f"R{r1+1}–R{r2+1}"
    kappa_pairs_val.append(k)
    kappa_pairs_lbl.append(lbl)
    print(f"  {lbl}: κ = {k:.4f}  ({interpret_kappa(k)})")

kappa_stab_mean = np.mean(kappa_pairs_val)
kappa_stab_std  = np.std(kappa_pairs_val)

print(f"\n  κ Stability (rata-rata) : {kappa_stab_mean:.4f} ± {kappa_stab_std:.4f}  "
      f"({interpret_kappa(kappa_stab_mean)})")
print(f"  κ Reliability           : {kappa_rel:.4f}  ({interpret_kappa(kappa_rel)})")

# Metrik per run
final_run_rows = []
for r_idx, preds in enumerate(final_all_preds):
    m = compute_metrics(GROUND_TRUTH, preds)
    final_run_rows.append({"Run": r_idx + 1, **m})
final_run_df = pd.DataFrame(final_run_rows)

print(f"\nMetrik per Run:")
print(final_run_df[["Run","accuracy","f1_m","f1_w",
                    "precision_m","recall_m"]].to_string(index=False))
print(f"\nRata-rata ± Std:")
for col, key in [("Accuracy","accuracy"),("F1-Macro","f1_m"),("F1-Weighted","f1_w")]:
    print(f"  {col:<14}: {final_run_df[key].mean():.4f} ± {final_run_df[key].std():.4f}")

In [ ]:
# =============================================================================
# TAHAP 5: VISUALISASI & EKSPOR
# =============================================================================
print("TAHAP 5: VISUALISASI & EKSPOR")

labels_in_test = sorted(set(GROUND_TRUTH))
rpt_str = classification_report(GROUND_TRUTH, final_preds,
                                 labels=labels_in_test, zero_division=0)
print(f"\nClassification Report (Majority Vote, {N_FINAL_RUNS} run):")
print(rpt_str)

# ── Plot 1: Confusion Matrix ──────────────────────────────────────────────────
cm = confusion_matrix(GROUND_TRUTH, final_preds, labels=labels_in_test)
fig, ax = plt.subplots(figsize=(14, 11))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=labels_in_test, yticklabels=labels_in_test, ax=ax)
ax.set_xlabel("Predicted", fontsize=11)
ax.set_ylabel("Actual", fontsize=11)
ax.set_title(f"Confusion Matrix — Zero-Shot (T={BEST_T}, Majority Vote {N_FINAL_RUNS}x)",
             fontweight="bold")
ax.set_xticklabels(ax.get_xticklabels(), rotation=40, ha="right", fontsize=8)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=8)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/ZS_confusion_matrix_final.png", dpi=150, bbox_inches="tight")
plt.show()
print("✔ Plot disimpan: ZS_confusion_matrix_final.png")

# ── Plot 2: F1 per kelas ──────────────────────────────────────────────────────
rpt_dict = classification_report(GROUND_TRUTH, final_preds,
                                  labels=labels_in_test,
                                  output_dict=True, zero_division=0)
f1_per_class = {k: v["f1-score"] for k, v in rpt_dict.items()
                if k in labels_in_test}
f1_series = pd.Series(f1_per_class).sort_values()

fig, ax = plt.subplots(figsize=(12, 6))
colors = ["#2ecc71" if v >= 0.6 else "#e67e22" if v >= 0.3 else "#e74c3c"
          for v in f1_series.values]
ax.barh(f1_series.index, f1_series.values, color=colors)
ax.axvline(0.6, color="green",  linestyle="--", alpha=0.5, label="F1=0.6")
ax.axvline(0.3, color="orange", linestyle="--", alpha=0.5, label="F1=0.3")
ax.set_xlabel("F1-Score")
ax.set_title(f"F1-Score per Kategori — Zero-Shot (T={BEST_T})", fontweight="bold")
ax.legend()
for i, v in enumerate(f1_series.values):
    ax.text(v + 0.01, i, f"{v:.3f}", va="center", fontsize=8)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/ZS_f1_per_kelas.png", dpi=150, bbox_inches="tight")
plt.show()
print("✔ Plot disimpan: ZS_f1_per_kelas.png")

# ── Plot 3: κ Stability per pasang run ───────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
bar_colors = ["#27ae60" if k >= 0.6 else "#e67e22" if k >= 0.4 else "#e74c3c"
              for k in kappa_pairs_val]
ax.bar(kappa_pairs_lbl, kappa_pairs_val, color=bar_colors, alpha=0.85)
ax.axhline(kappa_stab_mean, color="navy", linestyle="--",
           label=f"Rata-rata κ = {kappa_stab_mean:.4f}")
ax.set_ylim(0, 1.05)
ax.set_xlabel("Pasang Run")
ax.set_ylabel("Cohen's κ")
ax.set_title(f"κ Stability Pairwise — Zero-Shot (T={BEST_T}, {N_FINAL_RUNS}x run)",
             fontweight="bold")
ax.legend()
for i, k in enumerate(kappa_pairs_val):
    ax.text(i, k + 0.01, f"{k:.3f}", ha="center", fontsize=9)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/ZS_kappa_stability.png", dpi=150, bbox_inches="tight")
plt.show()
print("✔ Plot disimpan: ZS_kappa_stability.png")

# ── Ekspor CSV & XLSX ─────────────────────────────────────────────────────────
df_logs = pd.DataFrame(final_logs)
df_logs.to_excel(f"{OUTPUT_DIR}/ZS_detail_per_run.xlsx", index=False)
df_logs.to_csv(f"{OUTPUT_DIR}/ZS_hasil_prediksi.csv", index=False)

final_pred_df = df_test[["tasklist_id", COL_LABEL, COL_TEXT]].copy()
final_pred_df["pred_label"] = final_preds
final_pred_df["correct"]    = (final_pred_df[COL_LABEL] == final_pred_df["pred_label"])
final_pred_df.to_excel(f"{OUTPUT_DIR}/ZS_prediksi_final.xlsx", index=False)

rpt_rows = []
for lbl in labels_in_test:
    row = rpt_dict.get(lbl, {})
    rpt_rows.append({"kategori": lbl, **row})
pd.DataFrame(rpt_rows).to_excel(f"{OUTPUT_DIR}/ZS_classification_report.xlsx", index=False)

summary = {
    "model"           : LLM_MODEL,
    "pendekatan"      : "Zero-Shot",
    "best_temperature": BEST_T,
    "n_final_runs"    : N_FINAL_RUNS,
    "n_test"          : N_TEST,
    **{f"mv_{k}": v for k, v in mv_final.items()},
    "kappa_reliability"   : round(kappa_rel, 4),
    "kappa_stability_mean": round(kappa_stab_mean, 4),
    "kappa_stability_std" : round(kappa_stab_std, 4),
    "timestamp"       : datetime.now().isoformat(),
}
with open(f"{OUTPUT_DIR}/ZS_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print(f"""
RINGKASAN AKHIR — ZERO-SHOT PIPELINE

  [KONFIGURASI TERPILIH]
  Temperature        : {BEST_T}
  Pendekatan         : Zero-Shot (tanpa retrieval)

  [METRIK MAJORITY VOTE ({N_FINAL_RUNS}x run)]
  Accuracy           : {mv_final['accuracy']:.4f}
  F1-Macro           : {mv_final['f1_m']:.4f}
  F1-Weighted        : {mv_final['f1_w']:.4f}
  Precision-Macro    : {mv_final['precision_m']:.4f}
  Recall-Macro       : {mv_final['recall_m']:.4f}

  [KESTABILAN]
  κ Reliability      : {kappa_rel:.4f}  ({interpret_kappa(kappa_rel)})
  κ Stability (mean) : {kappa_stab_mean:.4f} ± {kappa_stab_std:.4f}  ({interpret_kappa(kappa_stab_mean)})

  [OUTPUT FILES — output_zeroshot/]
  ZS_ablasi_temperature.xlsx
  ZS_detail_per_run.xlsx
  ZS_prediksi_final.xlsx
  ZS_classification_report.xlsx
  ZS_summary.json
  ZS_confusion_matrix_final.png
  ZS_f1_per_kelas.png
  ZS_kappa_stability.png

✅  Pipeline Zero-Shot selesai.
""")